# VisDrone2019-DET - bang ket qua phu (generalization)

Muc tieu: **mot bang duy nhat**, cung dinh dang main result nhung tren dataset
thu hai, de chung minh pipeline khong chi hop voi PASCAL VOC.

| | |
|---|---|
| Dataset | VisDrone2019-DET, 6471 train / 548 val, 10 lop |
| Baseline | yolo26m train tu trong so COCO, 100 epoch |
| Ours | L1-norm prune 50% (div 8) + finetune 100 epoch, CWD tau=9 |
| Batch / imgsz / seed | 16 / 640 / 0 |

`imgsz=640` la muc chuan trong cac bai nen model tren VisDrone (FDM-YOLO,
YOLOv8n-ACW, cac bang YOLOv8n/s) nen so lieu so sanh duoc.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
   (can Internet: VisDrone 2.3 GB va `yolo26m.pt` deu tai tu mang)
2. Bam **Save & Run All**. Khong can Add Data o lan dau.
3. Moi phien tu dung o 10h. Phan Ket qua se bao con thieu gi ->
   Add Data output lan nay roi Save & Run All lai.
4. Uoc tinh **2-3 phien**: baseline ~7-8h, prune+CWD ~8-9h.

Xong thi gui lai `results/e2e_manifest.json` trong tab Output.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup (clone fork + cai deps)

In [ ]:
import os, shutil, subprocess, sys, pathlib

WORK = pathlib.Path("/kaggle/working")
REPO_DIR = WORK / "yolo"

# PHAI dung fork nay, KHONG duoc "pip install ultralytics": checkpoint sau khi
# prune duoc pickle voi ultralytics.nn.tasks_pruned.DetectionModelPruned - ban
# Ultralytics chinh thuc khong load duoc.
if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", "main",
                        "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode:
        print((r.stderr or "").strip())
        raise SystemExit("Clone that bai - repo phai dang o che do PUBLIC.")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

if not (REPO_DIR / "scripts" / "run_e2e.py").exists():
    raise SystemExit("Thieu scripts/run_e2e.py - hay push ban moi nhat len nhanh main.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", "requirements.txt"], check=False)
# Cai fork o che do editable: DDP sinh tien trinh con chay file tam ngoai repo,
# neu fork khong duoc cai thi con bao "No module named 'ultralytics'".
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                "--no-deps"], check=False)

import torch
import ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Cau hinh

In [ ]:
DATA   = "VisDrone.yaml"   # Ultralytics tu tai 2.3 GB ve /kaggle/working/datasets
TAG    = "vd50"            # tach khoi cac tag r30..r70 cua sweep tren VOC
RATIO  = 0.5               # dung dung ti le cua bang chinh tren VOC
EPOCHS = 100
BATCH  = 16
IMGSZ  = 640
# yolo26m.pt (COCO) lam diem khoi dau cho stage baseline. Day KHONG phai
# baseline VOC - VisDrone co 10 lop nen phai train baseline rieng.
PRETRAINED = "yolo26m.pt"

# Kaggle giet phien o 12h. Phai tu dung truoc do: neu de Kaggle giet thi lan chay
# bi danh dau failed va KHONG luu output -> mat sach last.pt -> mat ca phien.
STOP_AFTER_H = 10.0   # train tu dung (mem, co final_eval)
HARD_LIMIT_H = 11.0   # giet han neu dung mem khong kip

# DDP: fork da dua finetune/kd/kd_teacher/... vao ultralytics/cfg/default.yaml
# nen chung nam trong trainer.args va di duoc sang tien trinh con.
# (Truoc do chung chi la attribute cua object trainer -> con mat sach ->
#  trainer.py:478 kd_enabled=False -> train 100 epoch KHONG co CWD, khong bao loi.)
USE_DDP = True
DEVICE = "0,1" if (USE_DDP and torch.cuda.device_count() > 1) else "0"

print("data  :", DATA)
print("device:", DEVICE, "(DDP)" if "," in DEVICE else "(1 GPU)")
print("ratio :", RATIO, "->", TAG)

## 3. Resume

In [ ]:
# Lan chay dau se bao "Copy 0" - binh thuong.
# Tu lan 2: Add Data -> Your Work -> chon output lan truoc, notebook tu chep
# runs/ + weights/ + manifest ve roi chay tiep tu cho dang do.
import glob, shutil, json as _j

n = 0

# Manifest: GOP chu khong ghi de. Neu Add Data nhieu output (vd ca output VOC)
# ma ghi de thi manifest cuoi se xoa mat tien do.
merged = {}
mf = REPO_DIR / "results" / "e2e_manifest.json"
if mf.exists():
    merged.update(_j.loads(mf.read_text()))
for depth in range(1, 6):
    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/results/e2e_manifest.json"):
        merged.update(_j.loads(pathlib.Path(src).read_text())); n += 1
if merged:
    mf.parent.mkdir(parents=True, exist_ok=True)
    mf.write_text(_j.dumps(merged, indent=2, ensure_ascii=False))


# Neu gan NHIEU output (phien 1, phien 2, ...) thi phai chon ban co NHIEU epoch
# nhat. Lay "cai dau tien tim thay" la sai: thu tu glob khong xac dinh, co the
# chep nham ban cu va mat vai gio train ma khong he biet.
def _epochs(d):
    d = pathlib.Path(d)
    f = d / "results.csv"
    if f.exists():
        try:
            rows = [r for r in f.read_text().strip().splitlines()[1:] if r.strip()]
            if rows:
                return int(float(rows[-1].split(",")[0]))
        except Exception:
            pass
    # Chi tai moi last.pt ve (vd tu mot version bi Kaggle danh dau failed):
    # so epoch nam ngay trong checkpoint.
    last = d / "weights" / "last.pt"
    if last.exists():
        try:
            import torch as _torch
            ck = _torch.load(last, map_location="cpu", weights_only=False)
            ep = int(ck.get("epoch", -1))
            total = int((ck.get("train_args") or {}).get("epochs", 0) or 0)
            del ck
            return (ep + 1) if ep >= 0 else total
        except Exception:
            pass
    return -1


cands = {}
for depth in range(1, 6):
    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/runs/e2e/*"):
        cands.setdefault(pathlib.Path(src).name, []).append(src)
for name, srcs in sorted(cands.items()):
    best = max(srcs, key=_epochs)
    if len(srcs) > 1:
        print("   {}: co {} ban, chon ban {} epoch".format(name, len(srcs), _epochs(best)))
    dst = REPO_DIR / "runs" / "e2e" / name
    if not dst.exists():
        shutil.copytree(best, dst); n += 1
        print("   {}: {} epoch da train".format(name, _epochs(best)))

# Du phong cho truong hop TU TAI last.pt ve roi upload thanh dataset:
# chi can co thu muc ten baseline_VisDrone / finetune_<TAG> (kem weights/last.pt)
# nam dau do trong /kaggle/input, khong bat buoc dung cau truc yolo/runs/e2e/.
# Dung cho version bi Kaggle danh dau failed: file van tai ve duoc nhung
# "Add Data -> Your Work" khong gan version do.
for _name in ("baseline_VisDrone", "finetune_" + TAG):
    _dst = REPO_DIR / "runs" / "e2e" / _name
    if _dst.exists():
        continue
    extra = []
    for depth in range(1, 7):
        for c in glob.glob("/kaggle/input/" + "*/" * depth + _name):
            if pathlib.Path(c, "weights", "last.pt").exists():
                extra.append(c)
    if extra:
        best = max(extra, key=_epochs)
        shutil.copytree(best, _dst); n += 1
        print("   (thu cong) {}: {} epoch  <- {}".format(_name, _epochs(best), best))

for depth in range(1, 6):
    for src in glob.glob("/kaggle/input/" + "*/" * depth + "yolo/weights/*.pt"):
        dst = REPO_DIR / "weights" / pathlib.Path(src).name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst); n += 1

print("Copy", n, "muc tu lan chay truoc")
if mf.exists():
    for k, v in _j.loads(mf.read_text()).items():
        print("   {:<22} {}".format(k, v.get("weights", v.get("log", ""))))

## 4. Baseline -> Prune -> Finetune + CWD -> Val

In [ ]:
import json

# Chon stage theo tien do. stage_prune goi need(man, "baseline_VisDrone") va se
# thoat voi loi ro rang neu baseline chua xong - nhung bao truoc o day thi log
# de doc hon, va tranh chay prune khi chac chan chua the chay.
man = json.loads(mf.read_text()) if mf.exists() else {}
BKEY = "baseline_VisDrone"     # run_e2e.dkey(): baseline tach theo dataset

if BKEY not in man:
    stages = ["baseline"]
    print("Buoc nay: TRAIN BASELINE yolo26m tren VisDrone")
else:
    stages = ["prune", "finetune", "val"]
    print("Baseline da xong ->", man[BKEY]["weights"])
    print("Buoc nay: PRUNE {:.0%} -> FINETUNE + CWD -> VAL".format(RATIO))

cmd = [sys.executable, "scripts/run_e2e.py",
       "--stage", *stages,
       "--data", DATA, "--tag", TAG,
       "--prune-ratio", str(RATIO),
       "--pretrained", PRETRAINED,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--imgsz", str(IMGSZ),
       "--device", DEVICE, "--resume",
       "--stop-after-h", str(STOP_AFTER_H)]
print(" ".join(cmd), flush=True)

# Hai lop chan thoi gian:
#   1. --stop-after-h: train tu dung sau N gio, chay final_eval, thoat GON.
#   2. HARD_LIMIT_H duoi day: neu (1) khong kip (epoch dang chay qua lau, hoac treo)
#      thi giet han tien trinh. Ultralytics luu last.pt moi epoch nen cung chi
#      lam mat dung mot epoch.
# Muc dich chung: notebook phai KET THUC BINH THUONG truoc moc 12h cua Kaggle.
import signal, time as _t

t0 = _t.time()
proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), start_new_session=True)
try:
    proc.wait(timeout=HARD_LIMIT_H * 3600)
except subprocess.TimeoutExpired:
    print(chr(10) + "!! Qua {}h -> dung tien trinh de output kip luu.".format(HARD_LIMIT_H),
          flush=True)
    # Phai giet ca NHOM tien trinh: DDP con nam trong nhom do, giet moi tien trinh
    # cha se de lai con mo coi van giu GPU.
    def _kill(sig):
        try:
            os.killpg(os.getpgid(proc.pid), sig)
            return True
        except Exception:
            return False
    if not _kill(signal.SIGTERM):
        proc.terminate()
    try:
        proc.wait(timeout=180)
    except subprocess.TimeoutExpired:
        if not _kill(signal.SIGKILL):
            proc.kill()
        proc.wait(timeout=60)

print(chr(10) + "Tien trinh ket thuc sau {:.2f}h".format((_t.time() - t0) / 3600))

## 5. Ket qua

In [ ]:
import json

man = json.loads(mf.read_text()) if mf.exists() else {}
val = man.get("val@" + TAG, {})

print("Dataset:", DATA, " ratio:", RATIO)
print()

if val.get("baseline") and val.get("pruned"):
    b, o = val["baseline"], val["pruned"]
    print("| Model | Params (M) | AP50 | AP50-95 |")
    print("|---|---:|---:|---:|")
    print("| yolo26m (baseline) | {} | {} | {} |".format(
        b["params_M"], b["AP50"], b["AP50_95"]))
    print("| Ours (prune {:.0%} + CWD) | {} | {} | {} |".format(
        RATIO, o["params_M"], o["AP50"], o["AP50_95"]))
    print()
    print("  Delta AP50    {:+.2f}".format(o["AP50"] - b["AP50"]))
    print("  Delta AP50-95 {:+.2f}".format(o["AP50_95"] - b["AP50_95"]))
    print("  Giam params   {:.1f}%".format(100 * (1 - o["params_M"] / b["params_M"])))
    print()
    print("XONG. Gui lai: /kaggle/working/yolo/results/e2e_manifest.json")
else:
    # In tien do that de biet con thieu bao nhieu epoch.
    for name, what in (("baseline_VisDrone", "baseline"),
                       ("finetune_" + TAG, "finetune + CWD")):
        csv = REPO_DIR / "runs" / "e2e" / name / "results.csv"
        if csv.exists():
            rows = [r for r in csv.read_text().strip().splitlines()[1:] if r.strip()]
            ep = int(float(rows[-1].split(",")[0])) if rows else 0
            print("  {:<16} epoch {}/{}".format(what, ep, EPOCHS))
        else:
            print("  {:<16} chua bat dau".format(what))
    print()
    print("CHUA XONG - Add Data output lan nay roi Save & Run All lai.")